In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os

# Definir variable de entorno con la ruta al archivo de configuración del cliente.
os.environ["CLIENT_CONFIG_PATH"] = "/path/to/client_config.yaml"

# Definir variable de entorno con la ruta al archivo de credenciales de Google Cloud.
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/path/to/your/service_account_key.json"

In [ ]:
import joblib
from meridian.analysis import optimizer, summarizer, GCPClient

In [ ]:
# Cargar 'model.pkl'
model = "model.pkl"
mmm = joblib.load(model)

In [ ]:
# Crear cliente de GCP
gcp_client = GCPClient()

In [ ]:
mmm_summarizer = summarizer.Summarizer(mmm)
start_date_ = "2025-01-01"
end_date_ = "2025-09-01"

mmm_summarizer.output_model_results_summary(
    filename="mmm_report.html",
    filepath="./reports/Auto/",
    start_date=start_date_,
    end_date=end_date_,
)

# Guardar en GCS
gcp_client.upload_file_to_gcs(
    bucket_name="your-gcs-bucket-name",
    prefix='Reports/Auto',
    full_path="./reports/Auto/mmm_report.html",
)

In [ ]:
mmm_summarizer = summarizer.Summarizer(mmm)
start_date_ = "2025-01-01"
end_date_ = "2025-09-01"
start_date_cm_ = "2024-01-01"
end_date_cm_ = "2024-09-01"

mmm_summarizer.output_comparison_metrics_summary(
    filename="mmm_report_comparison_metrics.html",
    filepath="./reports/Auto/",
    start_date=start_date_,
    end_date=end_date_,
    start_date_cm=start_date_cm_,
    end_date_cm=end_date_cm_,
    bq_product_or_service="Auto"
)

# Guardar en GCS
gcp_client.upload_file_to_gcs(
    bucket_name="your-gcs-bucket-name",
    prefix='Reports/Auto',
    full_path="./reports/Auto/mmm_report_comparison_metrics.html",
)

# Cargar en BigQuery
for file in os.listdir("./reports/Auto/"):
    if file.endswith(".parquet"):
        parquet_path = os.path.join("./reports/Auto/", file)
        gcp_client.load_parquet_to_bq(
            parquet_path=parquet_path,
            table_id="your-project-id.your_dataset.your_table",
        )

In [ ]:
start_date_ = "2025-01-01"
end_date_ = "2025-09-01"

# Crear el optimizador desde el modelo entrenado
budget_optimizer = optimizer.BudgetOptimizer(mmm)

# Ejecutar la optimización con todos los parámetros configurables
optimization_results = budget_optimizer.optimize(
    use_posterior=True,
    selected_times=None,
    fixed_budget=True, # Mantenemos el presupuesto total fijo
    budget=None, # None = usa el gasto histórico total
    start_date = start_date_,
    end_date = end_date_,
    # Restricciones Ajustadas
    target_roi=None,
    target_mroi=None,
    gtol=0.0001,
    use_optimal_frequency=True,
    use_kpi=True,
    confidence_level=0.9,
    batch_size=3000
)

optimization_results.output_optimization_summary(
    filename="optimization_output.html",
    filepath="./reports/Auto/",
)

# Guardar en GCS
gcp_client.upload_file_to_gcs(
    bucket_name="your-gcs-bucket-name",
    prefix='Reports/Auto/',
    full_path="./reports/Auto/optimization_output.html",
)